In [2]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

giga_key = os.getenv("GIGA_KEY")

if not giga_key:

    raise ValueError("Ключ GIGA_KEY не найден в .env")

print("Ключ найден:", giga_key[:8], "...")

Ключ найден: MDE5ZGMz ...


In [4]:
llm = GigaChat(

    credentials=giga_key,

    scope="GIGACHAT_API_PERS",

    model="GigaChat",

    verify_ssl_certs=False,

    temperature=0.2,

    max_tokens=1000,

    timeout=60

)

response = llm.invoke("Привет! Ответь одним коротким предложением.")

print(response.content)

Привет! Коротко и по делу.


In [12]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число — целое число, соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:
"""
)

chain = basic_prompt | llm | StrOutputParser()

In [24]:


df = pd.read_csv("rental_26.csv", sep=";")

test_texts = {

    i + 1: text

    for i, text in enumerate(df["text"].head(15))

}
for number, text in test_texts.items():

    result = chain.invoke({"text": text})

    print(f"Заявка №{number}")

    print(f"Текст: {text}")

    print(f"Результат: {result}")

    print("---")


Заявка №1
Текст: Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийская деревня или рядом ,предложения в лс.
Результат: 2
---
Заявка №2
Текст: Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный эконом. 3 взрослых и 3 детей (2,9,11 лет). Строго с 20 по 30 июля
Результат: 6
---
Заявка №3
Текст: Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17.07 по 27.07.
Результат: 2
---
Заявка №4
Текст: Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28.06 не далеко от моря.
Результат: 3
---
Заявка №5
Текст: Добрый День!! Семья 4 человека, 2 взрослых, дети 10 и 3 года, ищем жилье, можно 3-х местный с доп.местом. С 1 июля поближе к морю и не дорого
😉
Результат: 4
---
Заявка №6
Текст: Здравствуйте. Интересует жильё в Лазаревском. 3е взрослых.
Со своим сан узлом и желательно с балконом. Не больше 10мин до моря.
По приемлемым ценам.
С 4 августа дней на 7-10
Результат: 3
---
Заявка №7
Текст: Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 августа, недалеко от м